# Evaluation and error analysis: reviewed version

**Review date: 4 October 2026.** This notebook reproduces saved English M-ABSA results and explains the term, category, sentiment, rare-label and NULL findings.

Run cells in order with the Phase 1 analysis environment. This folder includes `corpus/`, both indexes and predictions; no training or downloads are performed. The original Colab notebook is preserved in `result_analysis/archive/team_review/original_notebook.ipynb`.

Full review, definitions, limitations and the professor's proposed loss sweep: [detailed-analysis.md](detailed-analysis.md).
Current reporting protocol: [amendment 1.1](protocol-amendment-2026-10-04.md). The main diagnostic question is where extraction first fails.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if (ROOT / "phase1_analysis" / "analyze_phase1.py").is_file():
    ROOT = ROOT / "phase1_analysis"
if not (ROOT / "analyze_phase1.py").is_file():
    raise FileNotFoundError("Set ROOT to the phase1_analysis folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from analyze_phase1 import build_analysis, write_outputs, write_report

OUTPUT = ROOT / "result_analysis" / "current"
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 20)

def show(frame, digits=4):
    print(frame.round(digits).to_string(index=False))

## 1. Validate runs and reproduce metrics

Pair standard and weighted by **condition, held-out domain and seed**. Duplicate or unmatched keys stop the analysis. Local raw test sentences, retained gold, training vocabularies and saved scores must agree. All seven folds, including Restaurant, are consolidated in the LODO index exactly once.

The supplied original notebook used five in-domain seeds and only seed 13 for six held-out domains. This version uses the current local matrix and reports the actual seed count.

In [ ]:
analysis = build_analysis(ROOT)
write_outputs(analysis, OUTPUT)
write_report(analysis, ROOT / "detailed-analysis.md")
runs = analysis["runs"]
summary = analysis["summary"]
print(f"Validated runs: {len(runs)}")
print("Missing planned keys:", analysis["metadata"]["missing_run_keys"])
print("Unexpected keys:", analysis["metadata"]["unexpected_run_keys"])
show(summary[["setting", "held_out_domain", "model", "n", "seed_ids", "complete"]])

## 2. In-domain precision, recall and F1

Scores use a 0-1 scale. Mean and sample SD are across seeds, not sentence resampling. Weighting's observed recall gain must be read alongside precision and full-triplet F1.

In [ ]:
indomain = summary[summary["setting"] == "In-domain"]
show(indomain[["model", "n", "precision_mean", "recall_mean", "f1_mean", "f1_std"]])
paired = analysis["paired"]
show(paired[paired["setting"] == "In-domain"][["seed", "precision_change", "recall_change", "f1_standard", "f1_weighted", "f1_change"]])

In [ ]:
metrics = ["precision", "recall", "f1"]
fig, ax = plt.subplots(figsize=(8, 4.5))
for offset, model, color in [(-0.18, "standard", "#356a9a"), (0.18, "weighted", "#d48242")]:
    row = indomain.set_index("model").loc[model]
    bars = ax.bar([i + offset for i in range(3)], [row[f"{m}_mean"] for m in metrics],
                  width=0.36, yerr=[row[f"{m}_std"] for m in metrics], capsize=3,
                  label=f"{model} (n={int(row['n'])})", color=color)
    ax.bar_label(bars, fmt="%.3f", padding=6)
ax.set_xticks(range(3), ["Precision", "Recall", "Exact-triplet F1"])
ax.set_ylim(0, 0.55)
ax.set_ylabel("Mean score (error bars: sample SD)")
ax.set_title("In-domain precision and recall tradeoff")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "indomain_precision_recall.png", dpi=160)
plt.show()

## 3. One primary error per retained gold triplet

Assign **correct -> term -> category -> sentiment**, choosing the first failed component. If both category and sentiment are wrong, category is primary and independent flags preserve the sentiment failure. This labels an observed failure; it does not establish a sole cause.

The old `metrics.json` error counters overlap. Use the recomputed rows below for a partition. Spurious aspects and duplicate predictions are separate prediction-side counts.

In [ ]:
show(runs[runs["setting"] == "In-domain"].groupby("model")[["correct", "term", "category", "sentiment", "spurious_aspects"]].mean().reset_index(), 1)
conditional = analysis["conditional"]
show(conditional[conditional["setting"] == "In-domain"].groupby("model")[[
    "category_on_matched_term_accuracy", "sentiment_on_matched_term_accuracy",
    "sentiment_on_matched_term_and_category_accuracy"
]].mean().reset_index())
print("Conditional accuracies exclude missing terms. Per-run correct counts and denominators are in conditional.csv.")
print(); print("Primary error shares and conditional failure rates (different denominators):")
show(analysis["taxonomy_summary"][analysis["taxonomy_summary"]["setting"] == "In-domain"][[
    "model", "dominant_primary_error", "term_error_share", "category_error_share", "sentiment_error_share",
    "term_failure_rate", "category_failure_given_term", "sentiment_failure_given_term_category"
]])


## 4. Cross-domain performance beyond the full triplet

Keep **exact-triplet F1** as the primary task score. Add **term F1**, **term+sentiment F1**, **term+category F1**, and **known-category triplet F1 with coverage** as secondary diagnostics.

An exact term and correct sentiment remain useful even when the category cannot be predicted. Zero known-category support is shown as N/A. Scores use sentence-level surface strings; saved artifacts lack offsets, so these are not character-boundary or repeated-occurrence metrics.

In [ ]:
transfer = summary[summary["setting"] == "LODO"]
show(transfer[["held_out_domain", "model", "n", "f1_mean", "aspect_f1_mean", "aspect_sentiment_f1_mean", "known_category_f1_mean", "category_coverage_mean"]])
print("Projected metrics deduplicate after projection. Their differences are not additive error counts.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, model in zip(axes, ["standard", "weighted"]):
    table = transfer[transfer["model"] == model].sort_values("held_out_domain")
    x = list(range(len(table)))
    ax.bar([i - 0.18 for i in x], table["f1_mean"], width=0.36, label="Full triplet", color="#356a9a")
    ax.bar([i + 0.18 for i in x], table["aspect_sentiment_f1_mean"], width=0.36, label="Term + sentiment", color="#d48242")
    ax.set_xticks(x, table["held_out_domain"], rotation=35, ha="right")
    ax.set_title(f"{model}: component transfer")
    ax.set_ylim(0, 0.7)
    ax.legend()
axes[0].set_ylabel("Mean F1 across matched seeds")
fig.tight_layout()
fig.savefig(FIGURES / "crossdomain_component_f1.png", dpi=160)
plt.show()

## 5. Rare versus unseen categories

Use each fold's source-training counts: **unseen = 0; rare = 1-5; low = 6-20; medium = 21-100; frequent >100**. Count support and errors for every band. Do not reuse the in-domain rare set for cross-domain folds.

In [ ]:
rarity = analysis["rarity"]
rare_table = rarity[rarity["setting"] == "In-domain"].groupby(["model", "rarity"])[["gold_triplets", "correct", "term", "category", "sentiment"]].mean().reset_index()
rare_table["exact_triplet_recall"] = rare_table["correct"] / rare_table["gold_triplets"].replace(0, float("nan"))
show(rare_table)

## 6. NULL prevalence

Deduplicate annotations within each sentence. NULL targets retain their category and sentiment. The BIO architecture cannot produce them; report them separately from explicit-only errors.

The scope bound is **retained explicit gold / all deduplicated gold**. It is an annotation-scope upper bound; truncation and unseen categories can lower achievable recall further. Primary explicit-only F1 is unchanged.

In [ ]:
nulls = analysis["null_prevalence"]
show(nulls[nulls["split"] == "test"][["domain", "unique_triplets", "null_triplets", "null_percentage", "null_only_examples", "mixed_examples", "scope_recall_upper_bound"]], 3)
print("All 21 domain/split audits and NULL category/sentiment distributions are saved in null_prevalence.csv and null_labels.csv.")

## 7. Sentiment imbalance and Conflict

Preserve the original four-class comparison. A three-class sensitivity study needs a dated task change, filtered annotations and rebuilt training vocabularies/weights. Conflict must not be relabelled neutral, and deleting it from a static list is insufficient because vocabularies are derived from training data.

There are five deduplicated Conflict training annotations but only one retained explicit target; development and test have none. This cannot support an estimate of Conflict generalization.

In [ ]:
sentiment_counts = analysis["sentiment_counts"]
labels = ["positive", "negative", "neutral", "conflict"]
for scope in ["raw", "deduplicated", "retained"]:
    table = sentiment_counts[sentiment_counts["scope"] == scope].groupby(["split", "sentiment"])["count"].sum().unstack(fill_value=0).reindex(columns=labels).reset_index()
    print(); print(f"{scope} sentiment annotations:")
    show(table, 0)
confusion = analysis["sentiment_confusion"]
for model in ["standard", "weighted"]:
    subset = confusion[(confusion["setting"] == "In-domain") & (confusion["model"] == model)]
    matrix = subset.pivot_table(index="gold_sentiment", columns="predicted_sentiment", values="count", aggfunc="sum", fill_value=0).reindex(index=labels, columns=labels, fill_value=0)
    print(); print(f"{model}: sentiment confusion on matched terms, summed over seeds")
    print(matrix.to_string())
print("Confusion selects exact triplet, then same category, then lexicographic candidate. Missing terms are excluded; see run_diagnostics.json for denominators.")

## 8. Hardest domain

Rank the in-domain and LODO conditions separately. Full-triplet LODO scores tie at zero where category coverage is absent. Use component and coverage-aware rankings without claiming that associations prove causes.

In [ ]:
domain_metrics = analysis["domain_metrics"]
in_domain_rank = domain_metrics[domain_metrics["setting"] == "In-domain"].groupby(["domain", "model"])["f1"].mean().unstack().sort_values("standard").reset_index()
print("In-domain exact-triplet F1 ranking:")
show(in_domain_rank)
for model in ["standard", "weighted"]:
    print(); print(f"LODO {model}: ordered by term F1")
    show(transfer[transfer["model"] == model].sort_values("aspect_f1_mean")[["held_out_domain", "aspect_f1_mean", "aspect_sentiment_f1_mean", "f1_mean", "category_coverage_mean"]])

## 9. Professor's loss experiments - separate continuation

Correct the protocol implementation before training. Then use development data only for a small sweep:

- **CE + weighted CE:** normalized mixture `(1-alpha)*CE + alpha*weighted_CE`, with alpha = 0, 0.25, 1/3, 0.5, 0.75, 1.
- **Focal loss:** gamma = 1 and 2, initially without extra inverse-frequency weighting.
- Keep outer head multipliers and training settings fixed. `w1=1,w2=0.5` maps to alpha=1/3; `w1=0.5,w2=0.5` maps to alpha=1/2 after dividing by w1+w2. This normalization must be documented.
- The current continuation uses seeds 13 and 42 for the development pilot; promote selected designs to five matched seeds. Isolate heads in a second ablation.

The separate version-2 runner now implements these losses; see experiment-guide-2026-10-05.md in the main repository. This folder contains historical Phase 1 evidence. Test outputs reviewed here must not guide the sweep; subsequent reuse of these tests is exploratory.

References: [PyTorch weighted CE reduction](https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html); [Lin et al., focal loss](https://arxiv.org/abs/1708.02002).

Detailed findings and limits: [detailed-analysis.md](detailed-analysis.md). Reproducible tables and all gold diagnostic rows: `result_analysis/current/`.